# Image Embedding Inference

Use the image encoder of the ConceptCLIP model to run inference on the BloodMNIST dataset,
and save the resulting image embeddings as CSV files, split by dataset partition.

Output:
- `data/image_features_train.csv`
- `data/image_features_val.csv`
- `data/image_features_test.csv`

Row format: `id, label, split, f0, f1, ..., f{D-1}`

# 1 Prepare Dataset (BloodMNIST)

In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

import medmnist
from medmnist import INFO

# ── Output directory ──────────────────────────────────────────────
OUT_DIR = Path('./data')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── BloodMNIST dataset ──────────────────────────────────────────
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

common_tf = transforms.Compose([transforms.ToTensor()])   # → [0,1] float, CHW

ds_train = DataClass(split='train', size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)
ds_val   = DataClass(split='val',   size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)
ds_test  = DataClass(split='test',  size=224, as_rgb=True, mmap_mode='r', transform=common_tf, download=True)

print(f"train: {len(ds_train)}  val: {len(ds_val)}  test: {len(ds_test)}")

train_loader = DataLoader(ds_train, batch_size=64, shuffle=False, num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=64, shuffle=False, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=64, shuffle=False, num_workers=0)

# 2 Load ConceptCLIP Model

In [ ]:
from transformers import AutoModel, AutoProcessor
from huggingface_hub import login, whoami
import os

# Set the HF_TOKEN environment variable before running:
#   Windows: set HF_TOKEN=<your_token>
#   Linux/Mac: export HF_TOKEN=<your_token>
# Or run interactively: huggingface-cli login
hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)
print(whoami())

In [ ]:
model     = AutoModel.from_pretrained('JerrryNie/ConceptCLIP', trust_remote_code=True)
processor = AutoProcessor.from_pretrained('JerrryNie/ConceptCLIP', trust_remote_code=True)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
model  = model.to(device).eval()

# Data is already in [0,1]; disable rescale and keep normalize
processor.image_processor.do_rescale  = False
processor.image_processor.do_normalize = True

print('Device:', device)

# 3 Run Inference and Save Image Embeddings

In [ ]:
def encode_and_save(loader, split_name: str, out_path: Path, start_id: int = 0) -> int:
    """Run the image encoder on all images in a DataLoader and save CLS embeddings as a CSV.
    CSV columns: id, label, split, f0, f1, ..., f{D-1}
    id is a globally sequential index starting from start_id.
    Returns the starting id for the next split.
    """
    all_feats  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=split_name):
            pixel_values = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
            img_cls, _   = model.encode_image(pixel_values, normalize=True)  # [B, D]
            all_feats.append(img_cls.float().cpu().numpy())
            lbs = labels.view(-1).numpy().astype(np.int64)
            all_labels.append(lbs)

    feats  = np.concatenate(all_feats,  axis=0)  # [N, D]
    labels = np.concatenate(all_labels, axis=0)  # [N]
    N, D   = feats.shape

    feat_cols = [f'f{i}' for i in range(D)]
    df = pd.DataFrame(feats, columns=feat_cols)
    df.insert(0, 'split', split_name)
    df.insert(0, 'label', labels)
    df.insert(0, 'id',    np.arange(start_id, start_id + N, dtype=np.int64))

    df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved {split_name}: {out_path}  shape=({N}, {D})  id=[{start_id}, {start_id+N-1}]")
    return start_id + N


split_loaders = [
    ('train', train_loader),
    ('val',   val_loader),
    ('test',  test_loader),
]

next_id = 0
for split_name, loader in split_loaders:
    out_path = OUT_DIR / f'image_features_{split_name}.csv'
    next_id = encode_and_save(loader, split_name, out_path, start_id=next_id)

# 4 Quick Validation

In [ ]:
# for split_name in ['train', 'val', 'test']:
#     p = OUT_DIR / f'image_features_{split_name}.csv'
#     df = pd.read_csv(p, nrows=3)
#     print(f"{p.name}  rows≈{sum(1 for _ in open(p))-1}  cols={len(df.columns)}")
#     print(df.head(2).to_string(), '\n')